# 03 Models

Trains and scores all six runs from the README:

| Run | What it is |
|---|---|
| Baseline | Most popular beers |
| A | Baseline + bonus for Norwegian beers (same bonus as C) |
| B | BPR on all other countries |
| C | B + bonus for Norwegian beers |
| D | BPR on Nordic users only |
| E | BPR on other countries, same amount of data as D |

Every run is scored the same way on validation and test users, against two answer keys: all hidden beers (`all`, used everywhere) and only hidden beers the user rated 3.5 or higher (`liked`, a robustness check in 04). The results are saved to `results/` and shown in `04_results.ipynb`.

In [1]:
import os
from pathlib import Path

# Run from the project folder so `src` can be imported
if Path.cwd().name == "notebooks":
    os.chdir("..")

## Setup (shared by all runs)

In [2]:
import pandas as pd

from src.metrics import evaluate, user_scores
from src.models import popular_beers, recommend_popular
from src.split import load_split

train, history, hidden = load_split()

# Coverage is measured against every beer the models can recommend
n_beers = train["beer_id"].nunique()
GROUPS = ["validation", "test"]
# Settings are picked with all hidden beers. 'liked' is only used to check the results in 04
LIKED_RATING = 3.5
ANSWER_KEYS = {"all": hidden, "liked": hidden[hidden["rating"] >= LIKED_RATING]}
results = []
user_results = []

In [3]:
def score_run(run_name, recommendations):
    """Evaluate a run on validation and test users, with both answer keys, and add it to the results."""
    for answer_key, key_hidden in ANSWER_KEYS.items():
        for group in GROUPS:
            group_hidden = key_hidden[key_hidden["group"] == group]
            scores = evaluate(recommendations, group_hidden, n_beers)
            results.append({"run": run_name, "group": group, "answer_key": answer_key, **scores})
            # Saved per user for the hypothesis tests and failure analysis in 04
            user_results.append(user_scores(recommendations, group_hidden)
                                .assign(run=run_name, group=group, answer_key=answer_key))
    return pd.DataFrame(results).query("run == @run_name and answer_key == 'all'")

## Baseline: most popular beers

Rank beers by how many users outside Norway have rated them. Every user gets the same list, minus beers they already rated.

In [4]:
ranking = popular_beers(train)
baseline_recommendations = recommend_popular(ranking, history)
score_run("baseline", baseline_recommendations)

,run,group,answer_key,recall@10,ndcg@10,coverage@10
0,baseline,validation,all,0.009687,0.286706,0.000033
1,baseline,test,all,0.012526,0.332327,0.000033


The 10 most popular beers:

In [5]:
from src.data import load_beers

beer_info = load_beers().set_index("beer_id")[["beer_name", "style", "brewery_name"]]
beer_info.loc[ranking[:10]]

,beer_name,style,brewery_name
beer_id,,,
2360,Rochefort Trappistes 10,Abt/Quadrupel,Brasserie Rochefort
53,Chimay Bleue (Blue) / Grande Réserve,Belgian Strong Ale,Chimay
2530,St. Bernardus Abt 12,Abt/Quadrupel,St. Bernardus Brouwerij
835,Orval,Belgian Ale,Brasserie dOrval
4934,Westvleteren 12 (XII),Abt/Quadrupel,Westvleteren Abdij St. Sixtus
1434,Duvel,Belgian Strong Ale,Duvel Moortgat
1267,Guinness Draught,Dry Stout,St. Jamess Gate (Diageo)
5368,Tripel Karmeliet,Abbey Tripel,Brouwerij Bosteels (AB-InBev)
110932,Flying Dog Raging Bitch Belgian-Style IPA,India Pale Ale (IPA),Flying Dog Brewery


## Run B: BPR on all other countries

BPR learns which beers are liked by the same people. It is trained on all users outside Norway, plus the first 5 ratings of each Norwegian user.

We try three model sizes (`factors`) on the validation users and keep the best one.

In [6]:
from src.models import recommend_bpr, train_bpr

validation_hidden = hidden[hidden["group"] == "validation"]
# Each model takes a few minutes to train, so keep them for run C
bpr_models = {factors: train_bpr(train, history, factors=factors) for factors in [32, 64, 128]}
bpr_tuning = pd.DataFrame({
    factors: evaluate(recommend_bpr(bpr, history), validation_hidden, n_beers)
    for factors, bpr in bpr_models.items()
}).T
bpr_tuning

,recall@10,ndcg@10,coverage@10
32,0.012994,0.275092,0.001252
64,0.014014,0.299357,0.001210
128,0.014694,0.305037,0.001216


In [7]:
best_factors = bpr_tuning["ndcg@10"].idxmax()
best_bpr = bpr_models[best_factors]
score_run("B", recommend_bpr(best_bpr, history))

,run,group,answer_key,recall@10,ndcg@10,coverage@10
4,B,validation,all,0.014694,0.305037,0.001216
5,B,test,all,0.009972,0.251078,0.001216


## Run C: BPR + bonus for Norwegian beers

Same model as B, but Norwegian beers get a bonus: `score = BPR score + w * (beer is from Norway)`.

We try different values of `w` on the validation users and keep the best one. If the best `w` is 0, the bonus does not help.

In [8]:
HOME_BONUSES = [0, 0.1, 0.2, 0.5, 1.0]

bonus_tuning = pd.DataFrame({
    w: evaluate(recommend_bpr(best_bpr, history, home_bonus=w), validation_hidden, n_beers)
    for w in HOME_BONUSES
}).T
bonus_tuning

,recall@10,ndcg@10,coverage@10
0.0,0.014694,0.305037,0.001216
0.1,0.016868,0.328557,0.001179
0.2,0.018789,0.346782,0.001118
0.5,0.020662,0.377752,0.000842
1.0,0.020510,0.384988,0.000471


In [9]:
best_bonus = bonus_tuning["ndcg@10"].idxmax()
print("Best w:", best_bonus)
score_run("C", recommend_bpr(best_bpr, history, home_bonus=best_bonus))

Best w: 1.0


,run,group,answer_key,recall@10,ndcg@10,coverage@10
8,C,validation,all,0.02051,0.384988,0.000471
9,C,test,all,0.02061,0.359970,0.000471


## Run A: baseline + bonus for Norwegian beers

The popular list with the same bonus as C: `score = popularity + w * (beer is from Norway)`. Popularity is the number of ratings divided by the most ratings of any beer, so it goes from 0 to 1.

We use the `w` picked for C, so A and C only differ in popularity vs. BPR. This shows how much of C's gain comes from the bonus alone.

In [10]:
from src.split import TARGET_COUNTRY

bonus_ranking = popular_beers(train, TARGET_COUNTRY, home_bonus=best_bonus)
score_run("A", recommend_popular(bonus_ranking, history))

,run,group,answer_key,recall@10,ndcg@10,coverage@10
12,A,validation,all,0.013087,0.306722,0.000033
13,A,test,all,0.017058,0.309533,0.000033


The top 10 of run A:

In [11]:
beer_info.loc[bonus_ranking[:10]]

,beer_name,style,brewery_name
beer_id,,,
53853,Mikkeller Beer Geek Breakfast,Stout,Lervig Aktiebryggeri
97034,Mikkeller Beer Geek Brunch Weasel,Imperial Stout,Lervig Aktiebryggeri
49092,Nøgne Ø Imperial Stout,Imperial Stout,Nøgne Ø (Hansa Borg)
122174,Mikkeller Beer Geek Bacon,Smoked,Lervig Aktiebryggeri
118985,Mikkeller Beer Hop Breakfast,Black IPA,Lervig Aktiebryggeri
41915,Nøgne Ø # 100 (Batch 100),Barley Wine,Nøgne Ø (Hansa Borg)
214195,Mikkeller Beer Geek Vanilla Shake,Imperial Stout,Lervig Aktiebryggeri
125773,Nøgne Ø Two Captains Double IPA,Imperial IPA,Nøgne Ø (Hansa Borg)
124538,Nøgne Ø # 500 Imperial India Pale Ale (Batch 500),Imperial IPA,Nøgne Ø (Hansa Borg)


## Run D: BPR on Nordic users only

Same as B, but trained only on users from Sweden, Denmark and Finland (plus the first 5 Norwegian ratings).

In [12]:
from src.split import nordic_train

nordic_ratings = nordic_train(train)
print(f"{len(nordic_ratings):,} ratings from {nordic_ratings['user_id'].nunique():,} Nordic users")

nordic_bpr = train_bpr(nordic_ratings, history, factors=best_factors)
nordic_recommendations = recommend_bpr(nordic_bpr, history)
score_run("D", nordic_recommendations)

774,075 ratings from 1,951 Nordic users


,run,group,answer_key,recall@10,ndcg@10,coverage@10
16,D,validation,all,0.010453,0.275055,0.000505
17,D,test,all,0.009074,0.282118,0.000505


## Run E: BPR on other countries, same amount of data as D

A random sample from users outside the Nordics, with exactly as many ratings as D. Then D and E only differ in where the data comes from. Comparing them tests H3.

In [13]:
from src.split import non_nordic_sample

# Same number of ratings as D, so D and E only differ in where the data comes from
sampled_train = non_nordic_sample(train, len(nordic_ratings))
print(f"{len(sampled_train):,} ratings from {sampled_train['user_id'].nunique():,} non-Nordic users")

non_nordic_bpr = train_bpr(sampled_train, history, factors=best_factors)
non_nordic_recommendations = recommend_bpr(non_nordic_bpr, history)
score_run("E", non_nordic_recommendations)

774,075 ratings from 18,062 non-Nordic users


,run,group,answer_key,recall@10,ndcg@10,coverage@10
20,E,validation,all,0.008307,0.207171,0.001255
21,E,test,all,0.006877,0.174397,0.001255


## Save results

`model_results.csv` has the mean scores per run. `user_scores.csv` has the score of every user, used in `04_results.ipynb`.

In [14]:
pd.DataFrame(results).to_csv("results/model_results.csv", index=False)
pd.concat(user_results).to_csv("results/user_scores.csv", index=False)
pd.DataFrame(results)

,run,group,answer_key,recall@10,ndcg@10,coverage@10
0,baseline,validation,all,0.009687,0.286706,0.000033
1,baseline,test,all,0.012526,0.332327,0.000033
2,baseline,validation,liked,0.021940,0.249654,0.000033
3,baseline,test,liked,0.021635,0.264772,0.000033
4,B,validation,all,0.014694,0.305037,0.001216
5,B,test,all,0.009972,0.251078,0.001216
6,B,validation,liked,0.027604,0.189727,0.001216
7,B,test,liked,0.016512,0.152157,0.001216
8,C,validation,all,0.020510,0.384988,0.000471
9,C,test,all,0.020610,0.359970,0.000471
